# Prefix Caching & Chunked Prefill

> **Status:** Content notebook — experiments require a GPU runtime with vLLM installed.

## Learning Objectives

By the end of this notebook, you will be able to:

- [ ] Explain how prefix caching reduces TTFT for requests sharing a common prompt prefix
- [ ] Describe how chunked prefill prevents long-prompt requests from starving ongoing decode
- [ ] Enable and tune both features in vLLM
- [ ] Measure the impact of prefix caching on cache hit rate and latency
- [ ] Understand continuous batching and why it matters for real-time serving

---

## Prerequisites

- [03_kv_cache_paged_attention.ipynb](../03_kv_cache_paged_attention/03_kv_cache_paged_attention.ipynb)
- [02_serving_with_vllm.ipynb](../02_serving_with_vllm/02_serving_with_vllm.ipynb)

---

## 1. Prefix Caching

When multiple requests share the same prefix — e.g., a long system prompt, RAG-retrieved
documents, or few-shot examples — recomputing the KV cache for that prefix on every
request is wasteful.

**Prefix caching** stores KV blocks for repeated prefixes and reuses them across requests.

```
Request 1: [SYSTEM PROMPT (2048 tokens)] [User: "What is X?"]
                ↓
           KV blocks for system prompt cached with hash key

Request 2: [SYSTEM PROMPT (2048 tokens)] [User: "What is Y?"]
                ↓
           System prompt KV blocks reused → TTFT drops from ~500ms to ~50ms
```

### vLLM Automatic Prefix Caching

```bash
# Enable automatic prefix caching
python -m vllm.entrypoints.openai.api_server \
  --model meta-llama/Meta-Llama-3-8B-Instruct \
  --enable-prefix-caching \
  --max-model-len 8192
```

```python
# Test prefix cache hit rate
from openai import OpenAI
import time

client = OpenAI(base_url="http://localhost:8000/v1", api_key="none")

SYSTEM_PROMPT = "You are a helpful AI assistant. " * 200  # Long shared prefix

def query(user_msg: str) -> float:
    start = time.time()
    resp = client.chat.completions.create(
        model="meta-llama/Meta-Llama-3-8B-Instruct",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        max_tokens=50,
    )
    elapsed = time.time() - start
    return elapsed

# First request: no cache
t1 = query("What is 2+2?")
# Second request: prefix cached
t2 = query("What is 3+3?")
print(f"Cold start: {t1:.2f}s | Cached: {t2:.2f}s | Speedup: {t1/t2:.1f}x")
```

---

## 2. Continuous Batching

Traditional static batching waits for a full batch before starting inference.
This causes high latency for short requests that arrive when the batch is partially done.

**Continuous batching** allows new requests to join an in-progress batch at any
token boundary. vLLM uses continuous batching by default.

```
Static batching:        Continuous batching:
[R1][R2][R3][R4]       [R1          ]
Wait → Process → Done  [   R2       ]  ← R2 joins mid-stream
                       [      R3    ]  ← R3 joins mid-stream
                       [         R4]  ← R4 joins mid-stream
```

---

## 3. Chunked Prefill

**Problem:** A request with a 16K-token prompt monopolizes the GPU for its entire
prefill phase, causing all other requests (especially streaming ones) to wait.

**Chunked prefill** splits large prefill operations into smaller chunks (e.g., 2048
tokens per chunk), interleaving prefill chunks with ongoing decode steps.

```bash
# Enable chunked prefill with a 2048-token chunk size
python -m vllm.entrypoints.openai.api_server \
  --model meta-llama/Meta-Llama-3-8B-Instruct \
  --enable-prefix-caching \
  --enable-chunked-prefill \
  --max-num-batched-tokens 2048
```

Effect: Decode ITL (inter-token latency) variance decreases significantly for
mixed short/long request workloads.

---

## 4. Key Tuning Parameters

| Parameter | Effect | Typical value |
|-----------|--------|---------------|
| `--enable-prefix-caching` | Enable hash-based KV reuse | Always enable in prod |
| `--max-num-batched-tokens` | Chunk size for prefill | 512–4096 |
| `--gpu-memory-utilization` | KV cache budget | 0.85–0.95 |
| `--max-num-seqs` | Max concurrent sequences | 256–1024 |
| `--swap-space` | CPU KV swap buffer (GB) | 4–16 |

---

## 5. Measuring Cache Performance

vLLM exposes Prometheus metrics for monitoring cache performance:

```
# vLLM metrics endpoint
curl http://localhost:8000/metrics

# Key metrics to watch:
# vllm:cache_config_info          - Block size, total GPU/CPU blocks
# vllm:gpu_cache_usage_perc       - Fraction of GPU KV cache in use
# vllm:cpu_cache_usage_perc       - CPU swap usage (high = OOM risk)
# vllm:num_preemptions_total      - Requests evicted (should be near 0)
```

---

## Exercises

1. Run a benchmark with 100 requests sharing a 2048-token system prompt, with and
   without prefix caching. Report TTFT improvement.
2. Simulate a mixed workload: 50% short requests (256 tokens) and 50% long-context
   requests (8K tokens). Compare P95 TTFT with and without chunked prefill.
3. Graph GPU cache utilisation over time during a sustained load test.

---

## References

- [vLLM Prefix Caching docs](https://docs.vllm.ai/en/latest/features/automatic_prefix_caching.html)
- [Chunked Prefill explanation (vLLM blog)](https://blog.vllm.ai/2024/09/05/perf-update.html)
- [Sarathi-Serve paper (chunked prefill)](https://arxiv.org/abs/2308.16369)

## What Comes Next

- Return to the hub: [30-inference-optimization.ipynb](../30-inference-optimization.ipynb)
- Related: [14-local-llms/README.md](../../14-local-llms/README.md)
- Related: [09-mlops/README.md](../../09-mlops/README.md)
